# FreshMart Lab 0: Environment Verification
**Microsoft Fabric Data Science**

เป้าหมาย: ยืนยันว่าพร้อมเข้า Lab 1 — **ไม่ต้องวิเคราะห์ธุรกิจในแล็บนี้**

### ก่อนรัน (เช็ก 30 วินาที)
1. Workspace = ของคุณเอง (แนะนำชื่อ `labs`) — **ไม่ใช้ workspace ร่วม และไม่เชิญใครเข้า**
2. แนบ Default Lakehouse = **`lh_freshmart`** ที่สร้างเอง (ซ้ายมือของ notebook)
3. อัปโหลดไฟล์จาก `labs/data/` ไปที่ `Files/raw/` แล้ว
4. รอ Spark ขึ้น Ready (รอบแรก 1–2 นาทีได้ — **อย่ารันหลายเซลล์ซ้อน**)

### ศัพท์ที่ใช้ในแล็บนี้
Lakehouse คือที่เก็บไฟล์และตารางใน OneLake ชั้น Bronze คือข้อมูลดิบที่คุณสร้างจาก CSV ในแล็บนี้ เช่นตาราง `bronze.transactions`  
อธิบายเพิ่ม: ดูอภิธานศัพท์ใน repository ที่ `docs/glossary.md`

### ถ้าติด — อ่านก่อนถาม TA
| อาการ | ทำอะไร |
| --- | --- |
| `Spark table unavailable` แล้วโหลด CSV | กลับไปรันเซลล์สร้างตาราง Bronze ก่อน — ถ้ายังเห็นแถวครบ จึงผ่านจุดตรวจได้ |
| Kernel / Spark Starting ค้าง | รอ แล้วรันเซลล์เดิมอีกครั้งทีละเซลล์ |
| ไม่เจอไฟล์ใน `Files/raw/` | อัปโหลดสามไฟล์จาก `labs/data/` ตามคู่มือ Lab 0 — ไม่ต้องขอเข้า workspace ของผู้สอน |
| AssertionError จำนวนแถว | ไม่ผ่าน Lab 0 — อย่าข้ามไป Lab 1 |


### เซลล์ถัดไป: ฟังก์ชันโหลดข้อมูล

โค้ดด้านล่างนิยาม `load_table_or_csv` ให้แล้ว — **รันครั้งเดียวแล้วใช้ต่อทุก Lab**

- พยายามอ่านตาราง Lakehouse ก่อน (`bronze.transactions` ฯลฯ)
- ถ้าไม่มีตาราง จะอ่าน `Files/raw/*.csv` ให้อัตโนมัติ (ทางเลือกสำรอง)
- ไม่ต้องแก้โค้ดนี้


In [ ]:
from pathlib import Path
import pandas as pd

# Schema-qualified names (preferred). Legacy flat names still tried as fallback.
BRONZE_TRANSACTIONS = "bronze.transactions"
BRONZE_CUSTOMERS = "bronze.customers"
SILVER_CUSTOMER_FEATURES = "silver.customer_features"
GOLD_PREDICTIONS = "gold.freshmart_predictions"

def _first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None

def load_csv(file_name: str) -> pd.DataFrame:
    found = _first_existing([
        f"/lakehouse/default/Files/raw/{file_name}",
        f"Files/raw/{file_name}",
        f"../data/{file_name}",
        f"labs/data/{file_name}",
        file_name,
    ])
    if found is None:
        raise FileNotFoundError(f"Cannot find {file_name}. Upload it to Files/raw or place it under labs/data.")
    print(f"Loaded CSV: {found}")
    return pd.read_csv(found)

def _table_candidates(table_name: str) -> list[str]:
    legacy = {
        "bronze.transactions": "bronze_transactions",
        "bronze.customers": "bronze_customers",
        "silver.customer_features": "silver_customer_features",
        "gold.freshmart_predictions": "gold_freshmart_predictions",
    }
    names = [table_name]
    if table_name in legacy:
        names.append(legacy[table_name])
    return names

def load_table_or_csv(table_name: str, file_name: str) -> pd.DataFrame:
    last_error = None
    for candidate in _table_candidates(table_name):
        try:
            frame = spark.read.table(candidate).toPandas()
            print(f"Loaded Spark table {candidate}: {len(frame):,} rows")
            return frame
        except Exception as exc:
            last_error = exc
    print(f"Spark table '{table_name}' unavailable ({last_error}). Falling back to CSV.")
    return load_csv(file_name)


### สร้างตาราง Bronze จากไฟล์ดิบ

**โค้ดนี้ทำอะไร:** สร้าง schema `bronze` / `silver` / `gold` แล้วเขียนตาราง `bronze.transactions` และ `bronze.customers` จาก `Files/raw/`

รันครั้งเดียวหลังอัปโหลด CSV — รันซ้ำได้ (เขียนทับตาราง Bronze ชุดเดิม)

**สิ่งที่ควรเห็น**
- `bronze.transactions` ประมาณ **3,000** แถว
- `bronze.customers` ประมาณ **1,500** แถว
- ข้อความ `Bronze tables ready`

In [ ]:
from pathlib import Path

RAW = Path("/lakehouse/default/Files/raw")
required = [
    "freshmart_transactions.csv",
    "freshmart_customers.csv",
    "freshmart_scoring_batch.csv",
]
missing = [name for name in required if not (RAW / name).exists()]
if missing:
    raise FileNotFoundError(
        "ไม่พบไฟล์ใน Files/raw/: "
        + ", ".join(missing)
        + " — อัปโหลดจาก labs/data/ ตามคู่มือ Lab 0"
    )

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")


def load_raw_csv(file_name: str):
    return (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .csv(f"Files/raw/{file_name}")
    )


load_raw_csv("freshmart_transactions.csv").write.mode("overwrite").saveAsTable(
    "bronze.transactions"
)
load_raw_csv("freshmart_customers.csv").write.mode("overwrite").saveAsTable(
    "bronze.customers"
)

print("bronze.transactions:", spark.table("bronze.transactions").count())
print("bronze.customers:", spark.table("bronze.customers").count())
print("Bronze tables ready")

### โหลด Bronze แล้วดูตัวอย่างแถว

**โค้ดนี้ทำอะไร:** อ่านตารางสมาชิกและธุรกรรม แล้วพิมพ์จำนวนแถวและ `head(5)`

**สิ่งที่ควรเห็น**
- Transactions ประมาณ **3,000** แถว
- Customers ประมาณ **1,500** แถว

ชื่อตารางแบบ `bronze.xxx` หมายถึง schema.table (มาตรฐาน lakehouse แบบมี schema)


In [ ]:
df_tx = load_table_or_csv("bronze.transactions", "freshmart_transactions.csv")
df_cust = load_table_or_csv("bronze.customers", "freshmart_customers.csv")

print(f"Transactions rows: {len(df_tx):,}")
print(f"Customers rows: {len(df_cust):,}")
print("Transaction columns:", list(df_tx.columns))
print("Customer columns:", list(df_cust.columns))
display(df_tx.head(5))
display(df_cust.head(5))


### จุดตรวจอัตโนมัติ

**โค้ดนี้ทำอะไร:** ถ้าจำนวนแถวไม่ตรง จะ `raise AssertionError` และหยุด

- ผ่านแล้วจะพิมพ์ `Lab 0 verification passed`
- ไม่ผ่าน แปลว่าสภาพแวดล้อมยังไม่พร้อม — **ถาม TA พร้อมคัดลอกข้อความ error ทั้งบรรทัด**


In [ ]:
if len(df_tx) != 3000:
    raise AssertionError(
        f"คาดว่าธุรกรรม 3,000 แถว แต่ได้ {len(df_tx):,} — ตรวจ Files/raw หรือตาราง bronze.transactions"
    )
if len(df_cust) != 1500:
    raise AssertionError(
        f"คาดว่าสมาชิก 1,500 แถว แต่ได้ {len(df_cust):,} — ตรวจ Files/raw หรือตาราง bronze.customers"
    )
print("Lab 0 verification passed")
